# Silver layer - transit metrics

This step aggregates simulated transit telemetry events from the Bronze layer into route-level operational metrics using 5-minute event-time windows.

The Silver layer enriches telemetry signals with latency indicators (ingestion delay, late events, and clock skew) and produces aggregated metrics used by downstream Gold KPI tables.

In [0]:
from pyspark.sql import functions as F

_ = spark.sql("USE azure_streaming_mvp")

### Processing parameters

The parameters below control the aggregation window, recent event filtering, and late-event detection thresholds.

In [0]:
TRANSIT_WINDOW = "5 minutes"    # window size
LOOKBACK_MINUTES = 60           # filter recent events
LATE_THRESHOLD_SEC = 120        # late-event threshold

In [0]:
bronze_all = spark.table("bronze_events")

# Filter to simulated transit events
bronze_recent = (
    bronze_all
    .filter(F.col("source") == F.lit("sim_transit"))
    .filter(F.col("event_time_ts") >= (F.current_timestamp() - F.expr(f"INTERVAL 1 MINUTE * {LOOKBACK_MINUTES}")))
)

### Telemetry signal enrichment

Derive route identifiers and latency-related data quality signals (clock skew and late events) from raw Bronze telemetry events.

In [0]:
bronze_enriched = (
    bronze_recent
    .withColumn("route_id", F.col("attrs").getItem("route_id"))
    # Compute ingestion latency between event_time and ingest_time
    .withColumn(
        "ingest_delay_sec_raw",
        F.unix_timestamp("ingest_time_ts") - F.unix_timestamp("event_time_ts")
    )
    # Detect clock skew when event_time is ahead of ingest_time
    .withColumn("is_clock_skew", F.col("ingest_delay_sec_raw") < F.lit(0))
    # Clamp negative delays to zero so aggregation metrics remain stable
    .withColumn("ingest_delay_sec", F.greatest(F.col("ingest_delay_sec_raw"), F.lit(0)))
    # Flag late events based on configured latency threshold
    .withColumn("is_late_event", F.col("ingest_delay_sec") > F.lit(LATE_THRESHOLD_SEC))
)

In [0]:
# Windowed aggregation
silver_transit = (
    bronze_enriched
    .groupBy(
        F.window("event_time_ts", TRANSIT_WINDOW).alias("window"),
        F.col("metric"),
        F.col("route_id")
    )
    .agg(
        F.avg("value").alias("avg_value"),
        F.count(F.lit(1)).alias("n_events"),
        F.avg(F.col("ingest_delay_sec")).alias("avg_ingest_delay_sec"),
        F.sum(F.col("is_late_event").cast("int")).alias("n_late_events"),
        F.sum(F.col("is_clock_skew").cast("int")).alias("n_clock_skew")
    )
    .withColumn("late_event_rate", 
                F.when(F.col("n_events") == 0, F.lit(0.0))
                .otherwise(F.col("n_late_events") / F.col("n_events"))
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("metric"),
        F.col("route_id"),
        F.col("avg_value"),
        F.col("n_events"),
        F.col("avg_ingest_delay_sec"),
        F.col("n_late_events"),
        F.col("late_event_rate"),
        F.col("n_clock_skew")
    )
)

In [0]:
# Persist Silver output
(silver_transit.write.mode("overwrite").saveAsTable("silver_transit_metrics"))
display(spark.table("silver_transit_metrics").orderBy(F.col("window_start").desc(), F.col("route_id")).limit(20))

window_start,window_end,metric,route_id,avg_value,n_events,avg_ingest_delay_sec,n_late_events,late_event_rate,n_clock_skew
2026-03-08T15:40:00.000Z,2026-03-08T15:45:00.000Z,occupancy,B1,44.25,4,171.75,3,0.75,0
2026-03-08T15:40:00.000Z,2026-03-08T15:45:00.000Z,delay_sec,M1,465.5,4,196.75,3,0.75,0
2026-03-08T15:40:00.000Z,2026-03-08T15:45:00.000Z,occupancy,M1,16.0,1,267.0,1,1.0,0
2026-03-08T15:40:00.000Z,2026-03-08T15:45:00.000Z,occupancy,M2,78.0,1,258.0,1,1.0,0
2026-03-08T15:40:00.000Z,2026-03-08T15:45:00.000Z,occupancy,R10,5.0,1,81.0,0,0.0,0
2026-03-08T15:40:00.000Z,2026-03-08T15:45:00.000Z,occupancy,T1,1.0,1,286.0,1,1.0,0
2026-03-08T15:40:00.000Z,2026-03-08T15:45:00.000Z,delay_sec,T1,258.6666666666667,3,155.33333333333334,1,0.3333333333333333,0
2026-03-08T15:40:00.000Z,2026-03-08T15:45:00.000Z,occupancy,X3,20.0,1,167.0,1,1.0,0
2026-03-08T15:35:00.000Z,2026-03-08T15:40:00.000Z,occupancy,B2,65.66666666666667,3,429.3333333333333,3,1.0,0
2026-03-08T15:35:00.000Z,2026-03-08T15:40:00.000Z,delay_sec,B2,57.0,1,306.0,1,1.0,0


In [0]:
# Row count
spark.sql("SELECT COUNT(*) AS c FROM silver_transit_metrics").show()

# Latest window
spark.sql("""
SELECT MAX(window_end) AS latest_window_end
FROM silver_transit_metrics
""").show()

# Check duplicate keys (window_start, metric, route_id)
spark.sql("""
SELECT window_start, metric, route_id, COUNT(*) AS row_count
FROM silver_transit_metrics
GROUP BY window_start, metric, route_id
HAVING row_count > 1
""").show(truncate=False)

+---+
|  c|
+---+
|130|
+---+

+-------------------+
|  latest_window_end|
+-------------------+
|2026-03-08 15:45:00|
+-------------------+

+------------+------+--------+---------+
|window_start|metric|route_id|row_count|
+------------+------+--------+---------+
+------------+------+--------+---------+



In [0]:
# ---- Notebook completion signal ----
dbutils.notebook.exit("OK")